In [0]:
from pyspark.sql.functions import expr, regexp_replace, upper, col

df = (
    spark.read.format("excel")
    .option("inferSchema", "true")
    .option("sheetName", "법인 공유 data+Q-report data+CQIS")
    .option("dataAddress", "A1:D9999")
    .load("/Volumes/sandbox/z_yeswook_kim/v_yeswook_kim/jira1312.xlsx")
)

# 첫 행을 컬럼으로 승격
new_columns = df.first()
df = df.filter(df[0] != new_columns[0])

# 공백을 언더바로 변환한 컬럼명 생성
new_columns_underscore = [c.replace(" ", "_") for c in new_columns]
df = df.toDF(*new_columns_underscore)

df = df.withColumn(
    "mac_addr_origin",
    regexp_replace(
        upper(col("Mac_Address")),
        r"(.{2})(?!$)",
        r"$1:"
    )
)

df = df.withColumn(
    "mac_addr",
    expr("eic_data_ods.tlamp.hash_mac(mac_addr_origin)")
)

df.write.mode("overwrite").saveAsTable("adhoc.heds.heds_1312")

display(df)

In [0]:
%sql
create table if not exists adhoc.heds.heds_1312_cl1 as 
WITH b AS (
  SELECT DISTINCT mac_addr
  FROM adhoc.heds.heds_1312
),

w23 AS (
  SELECT
    a.mac_addr,
    MAX(a.accum_run_time) AS max_accum_run_time
  FROM eic_data_ods.tlamp.normal_log_webos23 a
  LEFT SEMI JOIN b
    ON a.mac_addr = b.mac_addr
  WHERE a.context_name = 'tvpowerd'
    AND a.date_ym BETWEEN '2024-09' AND '2026-02'
  GROUP BY a.mac_addr
),

w24 AS (
  SELECT
    a.mac_addr,
    MAX(a.accum_run_time) AS max_accum_run_time
  FROM eic_data_ods.tlamp.normal_log_webos24 a
  LEFT SEMI JOIN b
    ON a.mac_addr = b.mac_addr
  WHERE a.context_name = 'tvpowerd'
    AND a.date_ym BETWEEN '2024-09' AND '2026-02'
  GROUP BY a.mac_addr
),

unioned AS (
  SELECT * FROM w23
  UNION ALL
  SELECT * FROM w24
),

final_max AS (
  SELECT
    mac_addr,
    MAX(max_accum_run_time) AS max_accum_run_time
  FROM unioned
  GROUP BY mac_addr
)

SELECT
  b2.*,
  f.max_accum_run_time
FROM adhoc.heds.heds_1312 b2
LEFT JOIN final_max f
  ON b2.mac_addr = f.mac_addr;

In [0]:
%sql
select *
from adhoc.heds.heds_1312_cl1

In [0]:
%sql
create table if not exists adhoc.heds.heds_1312_cl1_v2 as 
WITH b AS (
  SELECT DISTINCT mac_addr
  FROM adhoc.heds.heds_1312
),

w23 AS (
  SELECT
    a.mac_addr,
    MAX(a.accum_run_time) AS max_accum_run_time
  FROM eic_data_ods.tlamp.normal_log_webos23 a
  LEFT SEMI JOIN b
    ON a.mac_addr = b.mac_addr
  WHERE 1=1
    and a.context_name in ('com.webos.service.micomservice', 'com.webos.service.panelcontrolle')
    AND a.date_ym BETWEEN '2024-09' AND '2026-02'
  GROUP BY a.mac_addr
),

w24 AS (
  SELECT
    a.mac_addr,
    MAX(a.accum_run_time) AS max_accum_run_time
  FROM eic_data_ods.tlamp.normal_log_webos24 a
  LEFT SEMI JOIN b
    ON a.mac_addr = b.mac_addr
  WHERE 1=1
    and a.context_name in ('com.webos.service.micomservice', 'com.webos.service.panelcontrolle')
    AND a.date_ym BETWEEN '2024-09' AND '2026-02'
  GROUP BY a.mac_addr
),

unioned AS (
  SELECT * FROM w23
  UNION ALL
  SELECT * FROM w24
),

final_max AS (
  SELECT
    mac_addr,
    MAX(max_accum_run_time) AS max_accum_run_time
  FROM unioned
  GROUP BY mac_addr
)

SELECT
  b2.*,
  f.max_accum_run_time
FROM adhoc.heds.heds_1312 b2
LEFT JOIN final_max f
  ON b2.mac_addr = f.mac_addr;

In [0]:
%sql
select *
from adhoc.heds.heds_1312_cl1_v2